In [39]:
import ee
import geemap.core as geemap
import pandas as pd
ee.Authenticate()
ee.Initialize(project='firedatatest')

In [40]:
#narrow our data to only look at colorado during specific dates
start_date = '2006-01-01'
end_date = '2026-01-01'
colorado = ee.FeatureCollection("TIGER/2018/States").filter("NAME == 'Colorado'").first().geometry() #selects colorado geometry from all the states
gridmet = ee.ImageCollection("IDAHO_EPSCOR/GRIDMET").filterBounds(colorado).filterDate(start_date, end_date) #only collect images from colorado
firms = ee.ImageCollection("FIRMS").filterBounds(colorado) #only collect images from colorado
srtm = ee.Image("USGS/SRTMGL1_003").clip(colorado) #clip elevation data to only be colorado (clip for image is like using filter for imageCollection)

In [41]:
#Dynamic data - changes every day
#helpful enviornmental conditions for fire prediction
precipitation = 'pr' #precipitation amount: 0-690.44 mm
max_temperature ='tmmx' #maximum temperature: 233.08-327.14 K
min_humidity = 'rmin' #minimum relative humidity: 0-100%
wind_speed = 'vs' #wind velocity at 10m: 0.14-29.13 m/s
potential_fuel = 'erc' #energy release component: 0-131.85 (fire danger index)

#fire data
fire_data = firms.select('T21') #brightness temperature: 300-509.29 K

#static data - does not change
elevation = srtm.select('elevation') #elevation: -10-6500 m
terrain = ee.Terrain.products(elevation).select(['elevation', 'slope', 'aspect']) #calculates slope, and aspect, adding it to the elevation data

In [42]:
#We need to order each pixel with a grid, so it can be identified by entry rather than being a conitnous image.
#Use the Universal Transvers Mercator projection, which perserves area and angles in Colorado (Colorado is perfectly in UTM zone 13N)
projection = ee.Projection('EPSG:32613').atScale(4000) #4 km resolution
grid = colorado.coveringGrid(projection) #creates a grid of pixels over colorado, each pixel is 4 km x 4 km

In [43]:
#function to combine all of the data into one image collection, so it can be easily used for ML
def combine_data(image):
    #get the date 
    date = image.date()

    #get the fire data for that date
    fire = fire_data.filterDate(date,date.advance(1, 'day')).max().unmask(0) #get hottest fire data for the entire day, setting nulls to 0 (no fire)

    #get the weather data for that date
    weather = image.select([precipitation, max_temperature, min_humidity, wind_speed, potential_fuel]) #get the weather data for that date
    
    #combine the weather, terrain, and fire data into one image
    combined = weather.addBands(terrain).addBands(fire)

    #when we reduce the image to the grid, we want to preserve the topogrophy, so get the max, mean and standard deviation to see if it is mountainous
    reducer = ee.Reducer.max().combine(ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True), sharedInputs=True)

    #reduce the combined image to our grid, grabbing the max of the data for each 4x4km section.
    final = combined.reduceRegions(collection=grid, reducer=reducer, scale=4000).map(lambda feature: feature.set('date', date.format('YYYY-MM-dd')).setGeometry(None)) #add the date as a property to each feature
    return final

In [ ]:
#asyncronously exports the data to a csv in google drive
#If we export it all at once the file will be too large, so split it into one file per year
for year in range(2006, 2027):
    #start at the beginning of the year, and end at the beginning of the next year to get all the data for that year
    start = f'{year}-01-01'
    end = f'{year+1}-01-01'
    
    #combine the data from a year for each day, and flatten it into one collection of features
    final_data = gridmet.filterDate(start,end).map(combine_data).flatten() 

    file_name = f'Colorado_Wildfire_Data_{year}'
    
    #Tell google to export all of this data to a csv in my google drive
    task = ee.batch.Export.table.toDrive(
        collection=final_data,
        description=file_name,
        folder='EarthEngineData', 
        fileFormat='CSV'
    )
    
    task.start()

In [47]:
task.status()



{'state': 'COMPLETED',
 'description': 'Colorado_Wildfire_Data_2026',
 'priority': 100,
 'creation_timestamp_ms': 1776272411765,
 'update_timestamp_ms': 1776306141637,
 'start_timestamp_ms': 1776306140505,
 'task_type': 'EXPORT_FEATURES',
 'destination_uris': ['https://drive.google.com/#folders/1fyrvvye8vtFsVe_NNszz-twqzqKXSPMZ'],
 'attempt': 1,
 'batch_eecu_usage_seconds': 0.33427080512046814,
 'id': 'OJVTUA6OK2KEYF4VOWGQ3SJZ',
 'name': 'projects/firedatatest/operations/OJVTUA6OK2KEYF4VOWGQ3SJZ'}